# Inference, checkpoints, and reproducibility

Saving weights, eval, no_grad, and reload checks.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

Weights may be insufficient without configuration, tokenizer/processor, and normalization. Resuming training also needs optimizer/scheduler and RNG state. eval changes dropout and some normalization behavior; no_grad independently disables graph recording.

We serialize a state_dict in memory, reload it into the same architecture, and compare outputs. This verifies a weight round trip, not full training resume. Agent Studio uses tools and models for inference; training belongs in the respective workspaces.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
import io
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Dropout(0.3), nn.Linear(8, 2)).eval()
x = torch.randn(3, 4)
with torch.no_grad():
    before = model(x)
buffer = io.BytesIO(); torch.save(model.state_dict(), buffer); buffer.seek(0)
restored = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Dropout(0.3), nn.Linear(8, 2))
restored.load_state_dict(torch.load(buffer, weights_only=True)); restored.eval()
with torch.no_grad():
    after = restored(x)
print('max reload error:', (before - after).abs().max().item())
assert torch.equal(before, after)
restored.train()
with torch.no_grad():
    print('train-mode dropout delta:', (restored(x) - restored(x)).abs().max().item())

## Try it yourself

List the artifacts needed for VLA export: architecture, weights, state/action normalization, coordinate order, image processor, and horizon. What happens if action order is wrong?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [PyTorch: saving and loading](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)